# Some quick start code for TUM Hackathon

In [1]:
from langchain.document_loaders import PyPDFLoader
from ai_eval.config import global_config as glob

filename = "Allplan_2020_Manual.pdf"

loader = PyPDFLoader(f"{glob.DATA_PKG_DIR}/{filename}")

raw_data = loader.load()

texts = [page.page_content for page in raw_data]

print(f"Number of docs: {len(texts)}")

/Users/qtf4195/tum-hackathon/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of docs: 319


## (Optional) Preprocess and load data: 

In [2]:
from ai_eval.resources.preprocessor import Preprocessor
from ai_eval.config import global_config as glob

filename = "Allplan_2020_Manual.pdf"

pre = Preprocessor()

docs = pre.fetch_documents(
    blob_path=f"{glob.DATA_PKG_DIR}/{filename}", source="local"
)

documents = pre.chunk_documents(documents=docs)

print(f"Number of processed document chunks: {len(documents)}")

2025-11-21 22:04:37,760 - ai_eval.resources.preprocessor - INFO - Processed /Users/qtf4195/tum-hackathon/data/Allplan_2020_Manual.pdf
2025-11-21 22:04:37,768 - ai_eval.resources.preprocessor - INFO - Using SentenceChunker for document chunking.


Finished 'fetch_documents' in 2.0845 secs


🦛 choooooooooooooooooooonk 100% • 319/319 docs chunked [00:02<00:00, 107.77doc/s] 🌱
2025-11-21 22:04:41,213 - ai_eval.resources.preprocessor - INFO - Chunked 319 pages into 319 chunks.


Finished 'chunk_documents' in 3.4565 secs
Number of processed document chunks: 319


## Get annotated data:

In [3]:
from ai_eval.services.file import JSONService
from ai_eval.config import global_config as glob

json = JSONService(path="generated_qa_data_tum.json", root_path=glob.DATA_PKG_DIR, verbose=True)

qa_data = json.doRead()
print(f"Number of evaluation data samples: {len(qa_data)}")

2025-11-21 22:04:41,225 - ai_eval.services.file - INFO - Read: /Users/qtf4195/tum-hackathon/data/generated_qa_data_tum.json


Number of evaluation data samples: 100


### Fit RAG model on the generated data and create evaluation dataset

In [4]:
from ai_eval.resources import eval_dataset_builder as eval

ground_truth_contexts = [item["context"] for item in qa_data]
sample_queries = [item["question"] for item in qa_data]
expected_responses = [item["answer"] for item in qa_data]

In [9]:
from google.auth import default
creds, project = default()
print(creds.service_account_email)

venture-rag@venture-hack.iam.gserviceaccount.com


Example: using Vertex AI models

In [6]:
from langchain_google_vertexai import ChatVertexAI, VertexAIEmbeddings

chat_model = ChatVertexAI(
                        project=glob.GCP_PROJECT,
                        model_name="gemini-2.5-flash",
                        temperature=0.1,
                        max_retries=2,
                    )

embedding_model = VertexAIEmbeddings(
                        project=glob.GCP_PROJECT,
                        model_name="text-embedding-005",
                    )

/Users/qtf4195/tum-hackathon/.venv/lib/python3.13/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils
/Users/qtf4195/tum-hackathon/.venv/lib/python3.13/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [8]:
from langchain_community.vectorstores import FAISS
from ai_eval.resources.rag_template import FAISSRAG

vectorstore = FAISS.from_documents(documents, embedding_model)

# 1. Create your RAG instance
rag = FAISSRAG(chat_model, documents, k=3, vectorstore=vectorstore)    # some vanilla example
# rag = TFIDFRAG(models.qa_generator, documents, k=3)                 # our (naive) hackathon baseline 

query = "What is Allplan?"

the_relevant_docs = rag.retrieve(question=query)

answer, relevant_docs = rag.answer(question=query)

PermissionDenied: 403 Permission 'aiplatform.endpoints.predict' denied on resource '//aiplatform.googleapis.com/projects/venture-hack/locations/us-central1/publishers/google/models/text-embedding-005@default' (or it may not exist). [reason: "IAM_PERMISSION_DENIED"
domain: "aiplatform.googleapis.com"
metadata {
  key: "resource"
  value: "projects/venture-hack/locations/us-central1/publishers/google/models/text-embedding-005@default"
}
metadata {
  key: "permission"
  value: "aiplatform.endpoints.predict"
}
]

In [ ]:
# 2. Create the builder with the RAG instance
builder = eval.EvalDatasetBuilder(rag)

# 3. Build the evaluation dataset
evaluation_dataset = builder.build_evaluation_dataset(
    input_contexts=ground_truth_contexts,
    sample_queries=sample_queries,
    expected_responses=expected_responses,
)

In [ ]:
from ai_eval.resources import deepeval_scorer as deep 

scorer = deep.DeepEvalScorer(evaluation_dataset)

results = scorer.calculate_scores()
print(results)

/Users/avosseler/Github/Team/tum-hackathon/.venv/lib/python3.13/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
2025-11-20 17:59:09,975 - ai_eval.resources.get_models - INFO - Using chat model: gemini-2.5-flash
2025-11-20 17:59:09,977 - ai_eval.resources.get_models - INFO - Using judge model: gemini-2.5-flash
2025-11-20 17:59:09,977 - ai_eval.resources.get_models - INFO - Using embedding model: text-embedding-005
2025-11-20 17:59:09,977 - ai_eval.resources.get_models - INFO - Models loaded successfully
2025-11-20 17:59:09,978 - ai_eval.resources.deepeval_scorer - INFO - Starting evaluation...
2025-11-20 17:59:09,979 - ai_eval.resources.deepeval_scorer - ERROR - Evaluation failed: If using 'test_cases', you must also provide 'met

Attempt 1 failed: If using 'test_cases', you must also provide 'metrics'..
Retrying...
Attempt 2 failed: If using 'test_cases', you must also provide 'metrics'..
Retrying...
Attempt 3 failed: If using 'test_cases', you must also provide 'metrics'..
Retrying...
{}


In [8]:
scorer.get_overall_metrics()

{'Answer Relevancy': 0.9149721667221666,
 'Faithfulness': 0.9780194805194804,
 'Contextual Recall': 0.92,
 'Contextual Precision': 0.8933333333333333,
 'Average Performance': 0.9265812451437451}

In [9]:
scorer.get_summary(save_to_file=True)

2025-11-20 01:05:10,490 - ai_eval.resources.deepeval_scorer - INFO - Summary of test results generated successfully!
2025-11-20 01:05:10,503 - ai_eval.services.file - INFO - JSON Service Output to File: /Users/avosseler/Github/Team/tum-hackathon/data/deepeval_results.json


{0: {'query': 'How do I sort documents in ascending or descending order in the software?',
  'actual_output': 'You can sort documents by clicking the title of a column.\n\n*   To sort documents in ascending order, click the column title.\n*   To sort documents in descending order, click the same column title again.\n\nAn arrow indicates which column is being sorted and whether sorting is in ascending (arrow points upward) or descending (arrow points downward) order.',
  'expected_output': 'You can sort documents by clicking the title of a column. Clicking it once sorts in ascending order, and clicking the same title again sorts in descending order. An arrow indicates the sorted column and direction.',
  'actual_context': ['Installation, Basics Structuring and managing data 251 Sorting documents You can sort the documents by clicking the title of a column. Click the column title to sort the documents in ascending order. Click the same column title again to sort the documents in descendi

I0000 00:00:1763597131.843614 4491286 ssl_transport_security.cc:1884] Handshake failed with error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED: unable to get local issuer certificate
